In [79]:
import os
import numpy as np
import pandas as pd
from scipy.stats import zscore
from sklearn.feature_selection import f_classif
from itertools import combinations

In [80]:
DATA_PATH = "../outputs"

# Archivos
df_temp_path = os.path.join(DATA_PATH, "features_temporal.parquet")
df_spa_path  = os.path.join(DATA_PATH, "features_spatial.parquet")
df_freq_path = os.path.join(DATA_PATH, "features_frequency.parquet")

df_temp = pd.read_parquet(df_temp_path)
df_spa   = pd.read_parquet(df_spa_path)
df_freq  = pd.read_parquet(df_freq_path)


Se han eliminado las columnas identificativas file y epoch

In [81]:
df_temp = df_temp.drop(columns=['epoch'])
df_spa = df_spa.drop(columns=['epoch'])
df_freq = df_freq.drop(columns=['epoch'])
y = df_spa['label']

## Propuesta 1


In [82]:
def remove_constant_features(df, feature_cols):
    constant_cols = [col for col in feature_cols if df[col].nunique() <= 1]
    selected_cols = [col for col in feature_cols if col not in constant_cols]
    print(f"Eliminadas {len(constant_cols)} features constantes")
    return selected_cols, constant_cols

def outliers_to_missing(df, feature_cols, z_thresh=3.0):
    df_clean = df.copy()
    n_outliers_per_feature = pd.Series(index=feature_cols, dtype=int)
    for col in feature_cols:
        if not np.issubdtype(df_clean[col].dtype, np.number):
            continue
        z = np.abs(zscore(df_clean[col], nan_policy='omit'))
        outliers = z > z_thresh
        n_outliers_per_feature[col] = np.sum(outliers)
        df_clean.loc[outliers, col] = np.nan
    return df_clean, n_outliers_per_feature

def impute_missing(df, feature_cols, skew_thresh=0.5):
    df_imputed = df.copy()
    summary = []
    for col in feature_cols:
        if not np.issubdtype(df_imputed[col].dtype, np.number):
            continue
        skew_val = df_imputed[col].skew()
        n_missing = df_imputed[col].isna().sum()
        if n_missing == 0:
            method = "none"
        elif abs(skew_val) <= skew_thresh:
            df_imputed[col] = df_imputed[col].fillna(df_imputed[col].mean())
            method = "mean"
        else:
            df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())
            method = "median"
        summary.append({"feature": col, "skew": skew_val, "missing": n_missing, "imputation": method})
    imputation_summary = pd.DataFrame(summary)
    return df_imputed, imputation_summary

def remove_low_importance_features(df, y, feature_cols, pval_thresh=0.05):
    """Elimina features con ANOVA F-test p>pval_thresh"""
    X = df[feature_cols].values
    f, p = f_classif(X, y)
    selected_cols = [col for col, pv in zip(feature_cols, p) if pv <= pval_thresh]
    removed_cols = [col for col in feature_cols if col not in selected_cols]
    return selected_cols, removed_cols

def remove_highly_correlated(df, feature_cols, corr_thresh=0.95):
    """Elimina features altamente correladas entre sí"""
    corr_matrix = df[feature_cols].corr().abs()
    to_remove = set()
    for i, col1 in enumerate(feature_cols):
        if col1 in to_remove:
            continue
        for col2 in feature_cols[i+1:]:
            if corr_matrix.loc[col1, col2] > corr_thresh:
                to_remove.add(col2)
    selected_cols = [c for c in feature_cols if c not in to_remove]
    removed_cols = list(to_remove)
    return selected_cols, removed_cols

In [83]:
def clean_and_select(df, label, feature_name):
    feature_cols = [c for c in df.columns if c not in [label, 'subject']]

    # 1️⃣ eliminar constantes
    feature_cols, const_cols = remove_constant_features(df, feature_cols)

    # 2️⃣ outliers → NaN
    df_clean, outliers_count = outliers_to_missing(df, feature_cols)
    print(f"{feature_name}: {outliers_count.sum()} outliers convertidos a NaN")

    # 3️⃣ imputación
    df_clean, impute_summary = impute_missing(df_clean, feature_cols)
    print(f"{feature_name} - resumen imputación:")
    display(impute_summary.head())

    # 4️⃣ ANOVA F-test
    selected_cols, removed_low = remove_low_importance_features(df_clean, df_clean[label], feature_cols)
    print(f"{feature_name}: Eliminadas {len(removed_low)} features por baja relación con label")

    # 5️⃣ Correlación
    selected_cols, removed_corr = remove_highly_correlated(df_clean, selected_cols)
    print(f"{feature_name}: Eliminadas {len(removed_corr)} features por alta correlación")

    # Mantener subject + label
    df_final = df_clean[selected_cols + ['subject', label]]
    return df_final


subject no se elimina

## Limpieza

In [84]:
df_temp_clean = clean_and_select(df_temp, "label", "Temporal")
df_spa_clean  = clean_and_select(df_spa, "label", "Espacial")
df_freq_clean = clean_and_select(df_freq, "label", "Frecuencial")

Eliminadas 0 features constantes
Temporal: 111044.0 outliers convertidos a NaN
Temporal - resumen imputación:


,feature,skew,missing,imputation
0,EEG_Pz_mean_6,-0.015468,536,mean
1,EEG_Pz_max_6,0.360806,527,mean
2,EEG_Pz_min_6,-0.377912,506,mean
3,EEG_Pz_auc_6,-0.016466,531,mean
4,EEG_Pz_mean_18,-0.023843,519,mean


Temporal: Eliminadas 177 features por baja relación con label
Temporal: Eliminadas 11 features por alta correlación
Eliminadas 0 features constantes
Espacial: 15193.0 outliers convertidos a NaN
Espacial - resumen imputación:


,feature,skew,missing,imputation
0,xd_comp0_mean,-0.009219,344,mean
1,xd_comp0_max,1.381393,348,median
2,xd_comp0_min,-1.459487,435,median
3,xd_comp0_auc,-0.009554,340,mean
4,xd_comp1_mean,0.022060,402,mean


Espacial: Eliminadas 22 features por baja relación con label
Espacial: Eliminadas 1 features por alta correlación
Eliminadas 14 features constantes
Frecuencial: 6131.0 outliers convertidos a NaN
Frecuencial - resumen imputación:


,feature,skew,missing,imputation
0,EEG_Pz_bp_alpha,2.219017,549,median
1,EEG_Pz_bp_beta,2.232833,552,median
2,EEG_Cz_bp_alpha,2.873833,503,median
3,EEG_Cz_bp_beta,2.441424,482,median
4,EEG_CPz_bp_alpha,2.356124,596,median


Frecuencial: Eliminadas 8 features por baja relación con label
Frecuencial: Eliminadas 0 features por alta correlación


In [85]:
df_temp_clean.to_parquet(os.path.join(DATA_PATH, "features_temporal_clean.parquet"), engine="fastparquet", index=False)
df_spa_clean.to_parquet(os.path.join(DATA_PATH, "features_spatial_clean.parquet"), engine="fastparquet", index=False)
df_freq_clean.to_parquet(os.path.join(DATA_PATH, "features_frequency_clean.parquet"), engine="fastparquet", index=False)

print("\nArchivos exportados con limpieza + selección de variables aplicada:")
print("features_temporal_clean.parquet")
print("features_spatial_clean.parquet")
print("features_frequency_clean.parquet")


Archivos exportados con limpieza + selección de variables aplicada:
features_temporal_clean.parquet
features_spatial_clean.parquet
features_frequency_clean.parquet
